# Equatorial Diurnal Cycle Benchmark

Reproduce the canonical Hayne et al. (2017) result for a dry-regolith
equatorial surface:

| Quantity | Hayne 2017 value | Tolerance |
|---|---|---|
| Daytime peak T | ~390 K | ±5 K |
| Nighttime minimum T | ~95 K | ±5 K |

Reference: Hayne, P. O. et al. (2017), JGR Planets 122, 2371-2400.
doi:10.1002/2017JE005387

The run uses:
- Hayne (2017) H-parameter density model (`density_hayne`)
- Martinez & Siegler (2021) conductivity with Woods-Robinson (2019) amorphous
  polynomial (`conductivity_martinez`)
- Biele (2022) rational specific-heat fit (`specific_heat(model='biele')`)
- Geometric depth grid, 10-lunation spin-up, radiative surface BC
- Bottom BC: geothermal flux Q_b = 0.018 W m⁻² (equatorial, Langseth 1976)

In [ ]:
from __future__ import annotations
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from lunar.grid import make_geometric_grid
from lunar.properties import conductivity_martinez, density_hayne, specific_heat
from lunar.constants import Q_B_EQUATORIAL, SIGMA_SB, EMISSIVITY_DEFAULT
from lunar.solver import PixelInputs, solve_pixel

print('Imports OK')

## 1  Set up the forcing

In [ ]:
# Lunar sidereal period
T_LUNAR = 27.321661 * 86400.0  # s
S0      = 1361.0               # W m^-2  (Kopp & Lean 2011)

# One lunation, 3600-s cadence
dt   = 3600.0
N_t  = int(T_LUNAR / dt) + 1
t_s  = np.linspace(0.0, T_LUNAR, N_t)

# Equatorial insolation: S0 * max(0, cos(2π t / T_LUNAR))
# The cos peak mimics sub-solar noon; cos < 0 is the lunar night.
phase      = 2.0 * np.pi * t_s / T_LUNAR
insolation = S0 * np.maximum(0.0, np.cos(phase))

print(f'Peak insolation: {insolation.max():.1f} W m^-2')
print(f'Day fraction: {(insolation > 0).mean():.3f}')

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t_s / 86400.0, insolation, color='orange')
ax.set_xlabel('Days into lunation')
ax.set_ylabel('Insolation [W m⁻²]')
ax.set_title('Equatorial insolation forcing')
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('eq_insolation.png', dpi=100)
plt.close(fig)
print('Saved eq_insolation.png')

## 2  Build the depth grid and property models

In [ ]:
# Geometric grid:  dz0 = 3 mm (sub-diurnal skin depth),  z_max = 10 m
grid = make_geometric_grid(z_max=10.0, dz0=0.003, growth=1.08)
print(f'Grid: {grid.n_layers} layers,  z[0] = {grid.z_mid[0]*100:.2f} cm,  z[-1] = {grid.z_mid[-1]:.2f} m')

# Wrap the property models for the solver's functional API
def K_func(T, z):
    return conductivity_martinez(T, z)   # Martinez & Siegler 2021

def rho_func(z):
    return density_hayne(z)              # Hayne 2017

def cp_func(T):
    return specific_heat(T, model='biele')  # Biele 2022

print('Property models:')
z_test = np.array([0.0, 0.01, 0.1, 1.0])
T_test = np.array([300.0, 280.0, 260.0, 250.0])
print(f'  rho(z=0..1m)  = {density_hayne(z_test)}')
print(f'  K(T=300K, z=0..1m) = {conductivity_martinez(T_test, z_test)}')
print(f'  cp(T=300K) = {specific_heat(np.array([300.0]), model="biele")[0]:.1f} J kg^-1 K^-1')

## 3  Run the solver

In [ ]:
inputs = PixelInputs(
    grid=grid,
    t=t_s,
    bc_mode='radiative',
    insolation=insolation,
    albedo=0.12,
    emissivity=EMISSIVITY_DEFAULT,
    Q_b=Q_B_EQUATORIAL,    # 0.018 W m^-2
    K_func=K_func,
    rho_func=rho_func,
    cp_func=cp_func,
    n_lunations_spinup=10,
    spinup_tol_K=0.01,
)

print('Running 10-lunation spin-up...')
out = solve_pixel(inputs)
print(f'Spin-up: {out.n_spinup_cycles} cycles,  converged={out.converged}')

## 4  Benchmark: peak and minimum surface temperature

In [ ]:
T_surface = out.T[0, :]   # top layer = surface proxy
T_peak    = float(T_surface.max())
T_min     = float(T_surface.min())

HAYNE_PEAK_K = 390.0
HAYNE_MIN_K  =  95.0
TOL          =   5.0

print(f'Peak surface temperature : {T_peak:.1f} K  (Hayne 2017: ~{HAYNE_PEAK_K} K)')
print(f'Min  surface temperature : {T_min:.1f} K  (Hayne 2017: ~{HAYNE_MIN_K} K)')
print()

peak_ok = abs(T_peak - HAYNE_PEAK_K) < TOL
min_ok  = abs(T_min  - HAYNE_MIN_K ) < TOL

print(f'PEAK  benchmark: {"PASS" if peak_ok else "FAIL"} (Δ = {T_peak - HAYNE_PEAK_K:+.1f} K, tol=±{TOL} K)')
print(f'MIN   benchmark: {"PASS" if min_ok  else "FAIL"} (Δ = {T_min  - HAYNE_MIN_K:+.1f} K, tol=±{TOL} K)')

## 5  Diurnal surface temperature curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: surface T over one lunation
ax = axes[0]
ax.plot(t_s / 86400.0, T_surface, color='firebrick', lw=1.5, label='This work')
ax.axhline(390.0, ls='--', color='gray', lw=1, label='Hayne 2017 peak')
ax.axhline( 95.0, ls=':',  color='gray', lw=1, label='Hayne 2017 min')
ax.set_xlabel('Days into lunation', fontsize=11)
ax.set_ylabel('Surface temperature [K]', fontsize=11)
ax.set_title('Equatorial surface T (equator, lon=0)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right: T vs depth at noon and midnight
ax = axes[1]
i_noon     = int(np.argmax(insolation))
i_midnight = int(np.argmin(insolation))
ax.plot(out.T[:, i_noon],     out.z * 100.0, '-',  color='orange', lw=2, label='Noon')
ax.plot(out.T[:, i_midnight], out.z * 100.0, '--', color='royalblue', lw=2, label='Midnight')
ax.invert_yaxis()
ax.set_xlabel('Temperature [K]', fontsize=11)
ax.set_ylabel('Depth [cm]', fontsize=11)
ax.set_title('Subsurface T profile', fontsize=12)
ax.set_ylim(200, 0)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

fig.suptitle('Equatorial Diurnal Benchmark — Hayne 2017', fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig('eq_diurnal_benchmark.png', dpi=150)
print('Saved eq_diurnal_benchmark.png')
plt.close(fig)

## 6  Thermal parameter profile

In [ ]:
# Thermal parameter Γ = sqrt(K ρ cp) / sqrt(ω) — controls diurnal amplitude
T_noon = out.T[:, i_noon]
K_noon = conductivity_martinez(T_noon, grid.z_mid)
rho_z  = density_hayne(grid.z_mid)
cp_z   = specific_heat(T_noon, model='biele')
omega  = 2.0 * np.pi / T_LUNAR
gamma  = np.sqrt(K_noon * rho_z * cp_z) / np.sqrt(omega)

fig, axes = plt.subplots(1, 3, figsize=(13, 5))

for ax, y, label, color in zip(
    axes,
    [rho_z, K_noon, gamma],
    ['Density ρ [kg m⁻³]', 'Conductivity K [W m⁻¹ K⁻¹]', 'Thermal parameter Γ [J m⁻² K⁻¹ s⁻¹/²]'],
    ['#2c7bb6', '#d7191c', '#1a9641'],
):
    ax.plot(y, grid.z_mid * 100.0, color=color, lw=1.5)
    ax.invert_yaxis()
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel('Depth [cm]', fontsize=9)
    ax.set_ylim(200, 0)
    ax.grid(True, alpha=0.3)

fig.suptitle('Regolith property profiles (noon, equatorial)', fontsize=12)
fig.tight_layout()
fig.savefig('eq_property_profiles.png', dpi=150)
print('Saved eq_property_profiles.png')
plt.close(fig)

## Summary

| Quantity | This solver | Hayne 2017 | Status |
|---|---|---|---|
| Peak surface T | see above | ~390 K | see PASS/FAIL above |
| Min surface T | see above | ~95 K | see PASS/FAIL above |

Any discrepancy > 5 K is most likely due to the coarse sinusoidal insolation
proxy (Hayne 2017 used a full orbital geometry). Replace `insolation` with the
SPICE-driven series from `notebook 03_spice_insolation.ipynb` for a more
accurate comparison.